In [1]:
from verl.trainer.ppo.ray_trainer import RayPPOTrainer

import os
import ray
import hydra
import torch
import numpy as np

from pprint import pprint
from omegaconf import OmegaConf, open_dict
from math import ceil
import uuid
from contextlib import contextmanager
from dataclasses import dataclass, field
from enum import Enum
from typing import Type, Dict
from copy import deepcopy
from collections import defaultdict
from functools import partial
from tqdm import tqdm
from codetiming import Timer
from verl.utils.tracking import Tracking
from uuid import uuid4

from verl import DataProto
from verl.trainer.main_ppo import get_custom_reward_fn,get_running_jobs, log_using_devices,get_current_job_id,get_available_devices
from verl.utils.fs import copy_to_local
from verl.protocol import collate_fn as batch_collate_fn
from verl.protocol import pad_dataproto_to_divisor, unpad_dataproto
from verl.single_controller.base import Worker
from verl.single_controller.ray import RayResourcePool, RayWorkerGroup, RayClassWithInitArgs
from verl.single_controller.ray.base import create_colocated_worker_cls
from verl.trainer.ppo import core_algos
from verl.trainer.ppo.metric_utils import compute_data_metrics, compute_throughout_metrics, compute_timing_metrics, reduce_metrics, bootstrap_metric, calc_maj_val
from verl.utils.seqlen_balancing import get_seqlen_balanced_partitions, log_seqlen_unbalance
from verl.utils.checkpoint.checkpoint_manager import find_latest_ckpt_path
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from verl.utils.tracking import ValidationGenerationsLogger
from torch.utils.data import RandomSampler, SequentialSampler
from torchdata.stateful_dataloader import StatefulDataLoader

/opt/conda/envs/ptca/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-06 12:31:58,417	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [11]:
os.environ["ENSURE_CUDA_VISIBLE_DEVICES"] = os.environ.get('CUDA_VISIBLE_DEVICES', '')
ray.init(runtime_env={
    'env_vars': {
        'TOKENIZERS_PARALLELISM': 'true',
        'NCCL_DEBUG': 'WARN',
        'VLLM_LOGGING_LEVEL': 'WARN',
        # 'RAY_EXPERIMENTAL_NOSET_ROCR_VISIBLE_DEVICES': '1',
    }
})


2025-05-06 12:39:11,725	INFO worker.py:1660 -- Connecting to existing Ray cluster at address: node-0:6379...
2025-05-06 12:39:11,753	INFO worker.py:1843 -- Connected to Ray cluster. View the dashboard at 127.0.0.1:8265 


Python version:,3.9.19
Ray version:,2.44.1
Dashboard:,http://127.0.0.1:8265


(pid=121533, ip=100.64.156.198) WARNING 05-06 12:39:27 rocm.py:33] `fork` method is not supported by ROCm. VLLM_WORKER_MULTIPROC_METHOD is overridden to `spawn` instead.
(WorkerDict pid=121533, ip=100.64.156.198) os.environ['CUDA_VISIBLE_DEVICES']: 0
(WorkerDict pid=121533, ip=100.64.156.198) os.environ['LOCAL_RANK']: 0
(WorkerDict pid=121533, ip=100.64.156.198) os.environ['CUDA_VISIBLE_DEVICES']: 0
(WorkerDict pid=121533, ip=100.64.156.198) os.environ['LOCAL_RANK']: 0
(pid=121716, ip=100.64.156.198) WARNING 05-06 12:39:33 rocm.py:33] `fork` method is not supported by ROCm. VLLM_WORKER_MULTIPROC_METHOD is overridden to `spawn` instead.
(pid=121718, ip=100.64.156.198) WARNING 05-06 12:39:33 rocm.py:33] `fork` method is not supported by ROCm. VLLM_WORKER_MULTIPROC_METHOD is overridden to `spawn` instead.
(WorkerDict pid=121716, ip=100.64.156.198) os.environ['CUDA_VISIBLE_DEVICES']: 1
(WorkerDict pid=121716, ip=100.64.156.198) os.environ['LOCAL_RANK']: 1
(WorkerDict pid=121716, ip=100.64.

(WorkerDict pid=121533, ip=100.64.156.198) Flash Attention 2.0 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen2ForTokenClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`
(WorkerDict pid=121533, ip=100.64.156.198) You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████| 17/17 [00:02<00:00,  7.95it/s]
(WorkerDict pid=121717, ip=100.64.156.198) Some weights of Qwen2ForTokenClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-32B and are newly initialized: ['score.bias']


(WorkerDict pid=121717, ip=100.64.156.198) Monkey patch _flash_attention_forward in transformers.integrations.flash_attention
(pid=252953) WARNING 05-06 12:39:34 rocm.py:33] `fork` method is not supported by ROCm. VLLM_WORKER_MULTIPROC_METHOD is overridden to `spawn` instead. [repeated 13x across cluster]
(WorkerDict pid=121722, ip=100.64.156.198) os.environ['CUDA_VISIBLE_DEVICES']: 7 [repeated 60x across cluster]
(WorkerDict pid=121722, ip=100.64.156.198) os.environ['LOCAL_RANK']: 7 [repeated 60x across cluster]


...

(WorkerDict pid=252952) wrap_policy: functools.partial(<function _or_policy at 0x7f87d3155040>, policies=[functools.partial(<function transformer_auto_wrap_policy at 0x7f87d314cee0>, transformer_layer_cls={<class 'transformers.models.qwen2.modeling_qwen2.Qwen2DecoderLayer'>})])
(WorkerDict pid=121719, ip=100.64.156.198) Monkey patch _flash_attention_forward in transformers.integrations.flash_attention [repeated 13x across cluster]
(WorkerDict pid=121533, ip=100.64.156.198) Qwen2ForCausalLM contains 32.76B parameters
(WorkerDict pid=121718, ip=100.64.156.198) Total steps: 349900, num_warmup_steps: 10
(WorkerDict pid=121718, ip=100.64.156.198) Actor use_remove_padding=True
(WorkerDict pid=121722, ip=100.64.156.198) wrap_policy: functools.partial(<function _or_policy at 0x7f5ea44b40d0>, policies=[functools.partial(<function transformer_auto_wrap_policy at 0x7f5ea44aaf70>, transformer_layer_cls={<class 'transformers.models.qwen2.modeling_qwen2.Qwen2DecoderLayer'>})]) [repeated 15x across c

Loading safetensors checkpoint shards:   0% Completed | 0/17 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   6% Completed | 1/17 [00:00<00:04,  3.55it/s]
Loading safetensors checkpoint shards:  12% Completed | 2/17 [00:00<00:04,  3.21it/s]
Loading safetensors checkpoint shards:   0% Completed | 0/17 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  94% Completed | 16/17 [00:05<00:00,  3.04it/s] [repeated 28x across cluster]
(WorkerDict pid=252945) 
(WorkerDict pid=121533, ip=100.64.156.198) 
  0%|          | 0/19 [00:00<?, ?it/s].156.198) 
Loading safetensors checkpoint shards: 100% Completed | 17/17 [00:05<00:00,  3.18it/s] [repeated 6x across cluster]
100%|██████████| 19/19 [00:12<00:00,  1.53it/s]
(WorkerDict pid=121533, ip=100.64.156.198) /opt/conda/envs/ptca/lib/python3.9/site-packages/torch/distributed/fsdp/fully_sharded_data_parallel.py:690: FutureWarning: FSDP.state_dict_type() and FSDP.set_state_dict_type() are being deprecated. Please use APIs, get_state_dict

(WorkerDict pid=121533, ip=100.64.156.198) kwargs: {'n': 1, 'max_new_tokens': 16384, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'repetition_penalty': 1.0, 'temperature': 1.0, 'top_k': -1, 'top_p': 1.0, 'ignore_eos': False}
(WorkerDict pid=121533, ip=100.64.156.198) After building sglang rollout, memory allocated (GB): 7.628451347351074, memory reserved (GB): 29.759765625
(WorkerDict pid=121533, ip=100.64.156.198) After building sharding manager, memory allocated (GB): 7.628451347351074, memory reserved (GB): 29.759765625
(WorkerDict pid=121533, ip=100.64.156.198) RCCL version 2.20.5+hip6.2 HEAD:45b618a+
(WorkerDict pid=121533, ip=100.64.156.198) WARNING 05-06 12:42:14 config.py:3432] Current VLLM config is not set. [repeated 4632x across cluster]
(WorkerDict pid=252952) kwargs: {'n': 1, 'max_new_tokens': 16384, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'repetition_penalty': 1.0, 'temperature': 1.0, 'top_k': -1, 'top_p': 1.0, 'ignore_eos': False}


(WorkerDict pid=252953) /opt/conda/envs/ptca/lib/python3.9/site-packages/torch/distributed/fsdp/fully_sharded_data_parallel.py:690: FutureWarning: FSDP.state_dict_type() and FSDP.set_state_dict_type() are being deprecated. Please use APIs, get_state_dict() and set_state_dict(), which can support different parallelisms, FSDP1, FSDP2, DDP. API doc: https://pytorch.org/docs/stable/distributed.checkpoint.html#torch.distributed.checkpoint.state_dict.get_state_dict .Tutorial: https://pytorch.org/tutorials/recipes/distributed_checkpoint_recipe.html .
(WorkerDict pid=252953)   warnings.warn(


(WorkerDict pid=121533, ip=100.64.156.198) [rank-0]: Loading from /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340/actor/model_world_size_16_rank_0.pt and /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340/actor/optim_world_size_16_rank_0.pt and /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340/actor/extra_state_world_size_16_rank_0.pt
(WorkerDict pid=121533, ip=100.64.156.198) [rank-0]: Loading from /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340/critic/model_world_size_16_rank_0.pt and /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340/critic/optim_world_size_16_rank_0.pt and /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340/critic/extra_state_world_size_16_rank_0.pt [repeated 16x across cluster]


In [12]:
config_path = '/home/aiscuser/verl/recipe/dapo/qwen2.5_32b_dapo_ppo.yaml'
config = OmegaConf.load(config_path)

In [13]:
from verl.utils.fs import copy_to_local
# print initial config
from pprint import pprint
from omegaconf import OmegaConf
pprint(OmegaConf.to_container(config, resolve=True))  # resolve=True will eval symbol values
OmegaConf.resolve(config)

# download the checkpoint from hdfs
local_path = copy_to_local(config.actor_rollout_ref.model.path)

# instantiate tokenizer
from verl.utils import hf_tokenizer, hf_processor
tokenizer = hf_tokenizer(local_path)
processor = hf_processor(local_path, use_fast=True)  # used for multimodal LLM, could be none

# define worker classes
if config.actor_rollout_ref.actor.strategy == 'fsdp':
    assert config.actor_rollout_ref.actor.strategy == config.critic.strategy
    from verl.workers.fsdp_workers import ActorRolloutRefWorker, CriticWorker
    from verl.single_controller.ray import RayWorkerGroup
    ray_worker_group_cls = RayWorkerGroup

elif config.actor_rollout_ref.actor.strategy == 'megatron':
    assert config.actor_rollout_ref.actor.strategy == config.critic.strategy
    from verl.workers.megatron_workers import ActorRolloutRefWorker, CriticWorker
    from verl.single_controller.ray.megatron import NVMegatronRayWorkerGroup
    ray_worker_group_cls = NVMegatronRayWorkerGroup

else:
    raise NotImplementedError

from verl.trainer.ppo.ray_trainer import ResourcePoolManager, Role

role_worker_mapping = {
    Role.ActorRollout: ray.remote(ActorRolloutRefWorker),
    Role.Critic: ray.remote(CriticWorker),
    Role.RefPolicy: ray.remote(ActorRolloutRefWorker)
}

global_pool_id = 'global_pool'
resource_pool_spec = {
    global_pool_id: [config.trainer.n_gpus_per_node] * config.trainer.nnodes,
}
mapping = {
    Role.ActorRollout: global_pool_id,
    Role.Critic: global_pool_id,
    Role.RefPolicy: global_pool_id,
}

# we should adopt a multi-source reward function here
# - for rule-based rm, we directly call a reward score
# - for model-based rm, we call a model
# - for code related prompt, we send to a sandbox if there are test cases
# - finally, we combine all the rewards together
# - The reward type depends on the tag of the data
if config.reward_model.enable:
    if config.reward_model.strategy == 'fsdp':
        from verl.workers.fsdp_workers import RewardModelWorker
    elif config.reward_model.strategy == 'megatron':
        from verl.workers.megatron_workers import RewardModelWorker
    else:
        raise NotImplementedError
    role_worker_mapping[Role.RewardModel] = ray.remote(RewardModelWorker)
    mapping[Role.RewardModel] = global_pool_id

reward_manager_name = config.reward_model.get("reward_manager", "naive")
if reward_manager_name == 'naive':
    from verl.workers.reward_manager import NaiveRewardManager
    reward_manager_cls = NaiveRewardManager
elif reward_manager_name == 'prime':
    from verl.workers.reward_manager import PrimeRewardManager
    reward_manager_cls = PrimeRewardManager
elif reward_manager_name == 'dapo':
    from verl.workers.reward_manager import DAPORewardManager
    reward_manager_cls = DAPORewardManager
else:

    raise NotImplementedError

compute_score = get_custom_reward_fn(config)
reward_fn = reward_manager_cls(tokenizer=tokenizer,
                                num_examine=0,
                                compute_score=compute_score,
                                reward_fn_key=config.data.reward_fn_key)

# Note that we always use function-based RM for validation
val_reward_fn = reward_manager_cls(tokenizer=tokenizer,
                                    num_examine=1,
                                    compute_score=compute_score,
                                    reward_fn_key=config.data.reward_fn_key)
resource_pool_manager = ResourcePoolManager(resource_pool_spec=resource_pool_spec, mapping=mapping)

trainer = RayPPOTrainer(config=config,
                        tokenizer=tokenizer,
                        processor=processor,
                        role_worker_mapping=role_worker_mapping,
                        resource_pool_manager=resource_pool_manager,
                        ray_worker_group_cls=ray_worker_group_cls,
                        reward_fn=reward_fn,
                        val_reward_fn=val_reward_fn)
trainer.init_workers()


{'actor_rollout_ref': {'actor': {'checkpoint': {'contents': ['model',
                                                             'optimizer',
                                                             'extra']},
                                 'clip_ratio': 0.2,
                                 'clip_ratio_high': 0.28,
                                 'clip_ratio_low': 0.2,
                                 'entropy_coeff': 0,
                                 'fsdp_config': {'fsdp_size': -1,
                                                 'optimizer_offload': True,
                                                 'param_offload': True,
                                                 'wrap_policy': {'min_num_params': 0}},
                                 'grad_clip': 1.0,
                                 'kl_loss_coef': 0.0,
                                 'kl_loss_type': 'low_var_kl',
                                 'loss_agg_mode': 'token-mean',
                               

[validate_config] All configuration checks passed successfully!
dataset len: 1791700
dataset len: 960
Size of train dataloader: 3499
Total training steps: 349900


In [5]:
log_file = f"/mnt/output/logs/{trainer.config.trainer.project_name}/{trainer.config.trainer.experiment_name}/{trainer.config.trainer.experiment_name}.log"
os.makedirs(os.path.dirname(log_file), exist_ok=True)
logger = Tracking(project_name=trainer.config.trainer.project_name,
                    experiment_name=trainer.config.trainer.experiment_name,
                    default_backend=trainer.config.trainer.logger,
                    config=OmegaConf.to_container(trainer.config, resolve=True))

wandb: Currently logged in as: viscent (viatage) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using LocalLogger is deprecated. The constructor API will change 


In [15]:
def load_checkpoint(self, global_step_folder: str):
    print(f'Load from checkpoint folder: {global_step_folder}')
    # set global step
    self.global_steps = int(global_step_folder.split('global_step_')[-1])

    print(f'Setting global step to {self.global_steps}')
    print(f'Resuming from {global_step_folder}')

    actor_path = os.path.join(global_step_folder, 'actor')
    critic_path = os.path.join(global_step_folder, 'critic')
    # load actor
    self.actor_rollout_wg.load_checkpoint(actor_path,
                                            del_local_after_load=self.config.trainer.del_local_ckpt_after_load)
    # load critic
    if self.use_critic:
        self.critic_wg.load_checkpoint(critic_path,
                                        del_local_after_load=self.config.trainer.del_local_ckpt_after_load)

    # load dataloader,
    # TODO: from remote not implemented yet
    dataloader_local_path = os.path.join(global_step_folder, 'data.pt')
    if os.path.exists(dataloader_local_path):
        dataloader_state_dict = torch.load(dataloader_local_path, weights_only=False)
        self.train_dataloader.load_state_dict(dataloader_state_dict)
    else:
        print(f"Warning: No dataloader state found at {dataloader_local_path}, will start from scratch")

trainer.load_checkpoint = load_checkpoint.__get__(trainer)


In [ ]:
checkpoint_dir = '/mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340'
trainer.load_checkpoint(checkpoint_dir)

Load from checkpoint folder: /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340
Setting global step to 340
Resuming from /mnt/blob/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_340


In [10]:
ray.shutdown()